# Session 32: Logistic Regression Classification

This notebook implements Logistic Regression for at-risk classification.
- 1 = at-risk (G3 < 10)
- 0 = successful (G3 >= 10)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Load data

In [ ]:
data_dir = Path("data/processed")
X_full = pd.read_parquet(data_dir / "X_full.parquet")
y = pd.read_parquet(data_dir / "y_full.parquet")
y = y["G3"] if "G3" in y.columns else y.iloc[:, 0]
print(f"Features: {X_full.shape}, Target: {len(y)}")

## 3. Train/test split and classification labels

In [ ]:
Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.20, random_state=42)
yc = (y < 10).astype(int)
yctr = yc.loc[ytr.index]
ycte = yc.loc[yte.index]
print("Training:", yctr.value_counts().to_dict())
print("Test:", ycte.value_counts().to_dict())

## 4. Fit Logistic Regression

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf.fit(Xtr_f, yctr)
y_pred = clf.predict(Xte_f)
y_proba = clf.predict_proba(Xte_f)[:, 1]

print("Accuracy:", accuracy_score(ycte, y_pred))
print("Precision:", precision_score(ycte, y_pred))
print("Recall:", recall_score(ycte, y_pred))
print("F1:", f1_score(ycte, y_pred))
print("ROC_AUC:", roc_auc_score(ycte, y_proba))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(ycte, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Successful", "At-risk"])
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 6. Classification Report

In [ ]:
print(classification_report(ycte, y_pred, target_names=["Successful", "At-risk"]))

## 7. Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": Xtr_f.columns,
    "Coefficient": clf.named_steps["logisticregression"].coef_[0]
})
coef_df["Abs"] = coef_df["Coefficient"].abs()
display(coef_df.sort_values("Abs", ascending=False).head(10))

## 8. Classification Row

In [ ]:
tn, fp, fn, tp = cm.ravel()
classification_row = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(ycte, y_pred),
    "Precision": precision_score(ycte, y_pred),
    "Recall": recall_score(ycte, y_pred),
    "F1": f1_score(ycte, y_pred),
    "ROC_AUC": roc_auc_score(ycte, y_proba),
    "True_Negative": tn,
    "False_Positive": fp,
    "False_Negative": fn,
    "True_Positive": tp
}])
display(classification_row)

## Session 32 Complete

✅ Logistic Regression with StandardScaler
✅ Classification metrics computed
✅ Confusion matrix displayed
✅ Classification row created

## Session 33: KNN, SVM, and Naive Bayes Classification

This section adds three classifiers: K-Nearest Neighbors, Support Vector Machine, and Gaussian Naive Bayes.
All classifiers use the same training and test data with scaling for KNN and SVM.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Verify prerequisites
for obj in ["Xtr_f", "Xte_f", "yctr", "ycte"]:
    if obj not in globals():
        raise NameError(f"Missing required object: {obj}")

print("Session 33 prerequisites verified")
print("Training shape:", Xtr_f.shape)
print("Test shape:", Xte_f.shape)


In [ ]:
def evaluate_classifier(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
    }
print("Evaluator ready")


In [ ]:
# Train KNN, SVM, and Gaussian Naive Bayes
results = []
for code, name, family, clf in [
    ("KNN", "K-Nearest Neighbors", "Instance-based", KNeighborsClassifier()),
    ("SVM", "Support Vector Machine", "Maximum-margin", SVC(probability=True, random_state=42)),
    ("NB", "Gaussian Naive Bayes", "Probabilistic", GaussianNB()),
]:
    pipeline = make_pipeline(StandardScaler(), clf)
    pipeline.fit(Xtr_f, yctr)
    predictions = pipeline.predict(Xte_f)
    probabilities = pipeline.predict_proba(Xte_f)[:, 1]
    metrics = evaluate_classifier(ycte, predictions, probabilities)
    results.append({
        "Model": code,
        "Full_Model_Name": name,
        "Model_Family": family,
        "Scaling_Used": True,
        **metrics
    })
    print(f"{code} completed - F1: {metrics["f1"]:.4f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df.insert(0, "Session33_F1_Rank", range(1, len(results_df) + 1))

print("\nSession 33 Classification Results:")
print(results_df.to_string())


In [ ]:
# Save results to CSV
from pathlib import Path
repo_root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists()), Path.cwd())
output_dir = repo_root / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "session33_classification_rows.csv", index=False)
print("Results saved to:", output_dir / "session33_classification_rows.csv")


### Session 33 Interpretation

- **KNN** classifies based on similarity in scaled feature space
- **SVM** finds a maximum-margin decision boundary
- **Gaussian Naive Bayes** assumes conditional independence of predictors

Naive Bayes provides a useful baseline even with the independence assumption.


# Session 36: Gradient Boosting and AdaBoost
**GitHub deliverable:** Add Gradient Boosting and AdaBoost classifiers to the classification notebook.

In [ ]:
# SESSION_36_BOOSTING_GITHUB_DELIVERABLE
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

required_session36_variables = ["Xtr_f", "Xte_f", "yctr", "ycte"]
missing_session36_variables = [v for v in required_session36_variables if v not in globals()]
if missing_session36_variables:
    print("Note: Run earlier classification cells first to define:", missing_session36_variables)


In [ ]:
boosting_models_s36 = {"GradBoost": GradientBoostingClassifier(random_state=42), "AdaBoost": AdaBoostClassifier(random_state=42)}
print("Session 36 Boosting Models:", list(boosting_models_s36.keys()))


In [ ]:
if "classification_leaderboard" not in globals():
    classification_leaderboard = pd.DataFrame()
print("Leaderboard initialized for Session 36.")


<!-- SESSION 37 AUTOMATION CELL -->
# Session 37: Neural-Network Classification

This section trains a scaled multilayer perceptron (MLP) classifier using the existing classification split. It adds exactly one **MLP Classifier** row to the classification leaderboard, compares the MLP with the boosting models, and evaluates stability across five random seeds.

Project class definitions: **0 = at risk** and **1 = successful**.

In [ ]:
# SESSION 37 AUTOMATION CELL - imports and prerequisite checks
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

required_objects_s37 = ["Xtr_f", "Xte_f", "yctr", "ycte"]
missing_objects_s37 = [name for name in required_objects_s37 if name not in globals()]
if missing_objects_s37:
    raise NameError(
        "Run the earlier classification cells first. Missing objects: "
        + ", ".join(missing_objects_s37)
    )

yctr_s37 = np.asarray(yctr).ravel()
ycte_s37 = np.asarray(ycte).ravel()
if Xtr_f.shape[0] != len(yctr_s37) or Xte_f.shape[0] != len(ycte_s37):
    raise ValueError("Feature and target row counts do not match.")

observed_classes_s37 = set(np.unique(np.concatenate([yctr_s37, ycte_s37])))
if observed_classes_s37 != {0, 1}:
    raise ValueError(f"Expected binary labels 0 and 1; found {sorted(observed_classes_s37)}")
if len(np.unique(ycte_s37)) != 2:
    raise ValueError("The test set must contain both classes for ROC AUC.")

print("Session 37 prerequisites verified.")
print("Training shape:", Xtr_f.shape, "Test shape:", Xte_f.shape)

In [ ]:
# SESSION 37 AUTOMATION CELL - train and evaluate the required MLP
mlpc = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=1000,
        random_state=42,
    ),
)

with warnings.catch_warnings(record=True) as mlp_warning_records_s37:
    warnings.simplefilter("always", ConvergenceWarning)
    mlp_fit_start_s37 = time.perf_counter()
    mlpc.fit(Xtr_f, yctr_s37)
    mlp_fit_time_s37 = time.perf_counter() - mlp_fit_start_s37

mlp_model_s37 = mlpc.named_steps["mlpclassifier"]
mlp_convergence_warnings_s37 = sum(
    issubclass(item.category, ConvergenceWarning)
    for item in mlp_warning_records_s37
)
mlp_train_predictions_s37 = mlpc.predict(Xtr_f)
mlp_predictions_s37 = mlpc.predict(Xte_f)

class_positions_s37 = np.where(mlp_model_s37.classes_ == 0)[0]
if len(class_positions_s37) != 1:
    raise ValueError("The fitted MLP does not contain at-risk class 0.")
mlp_at_risk_probability_s37 = mlpc.predict_proba(Xte_f)[:, int(class_positions_s37[0])]

mlp_accuracy_s37 = accuracy_score(ycte_s37, mlp_predictions_s37)
mlp_precision_s37 = precision_score(ycte_s37, mlp_predictions_s37, pos_label=1, zero_division=0)
mlp_recall_s37 = recall_score(ycte_s37, mlp_predictions_s37, pos_label=1, zero_division=0)
mlp_f1_s37 = f1_score(ycte_s37, mlp_predictions_s37, pos_label=1, zero_division=0)
mlp_roc_auc_s37 = roc_auc_score((ycte_s37 == 0).astype(int), mlp_at_risk_probability_s37)
mlp_at_risk_precision_s37 = precision_score(ycte_s37, mlp_predictions_s37, pos_label=0, zero_division=0)
mlp_at_risk_recall_s37 = recall_score(ycte_s37, mlp_predictions_s37, pos_label=0, zero_division=0)
mlp_at_risk_f1_s37 = f1_score(ycte_s37, mlp_predictions_s37, pos_label=0, zero_division=0)

if "eval_clf" in globals():
    print("Project eval_clf result:", eval_clf(ycte_s37, mlp_predictions_s37))

mlp_row_s37 = pd.DataFrame([{
    "Model": "MLP Classifier",
    "Accuracy": mlp_accuracy_s37,
    "Precision": mlp_precision_s37,
    "Recall": mlp_recall_s37,
    "F1": mlp_f1_s37,
    "ROC AUC": mlp_roc_auc_s37,
    "At-Risk Precision": mlp_at_risk_precision_s37,
    "At-Risk Recall": mlp_at_risk_recall_s37,
    "At-Risk F1": mlp_at_risk_f1_s37,
    "Fit Time Seconds": mlp_fit_time_s37,
}])
display(mlp_row_s37.round(4))

In [ ]:
# SESSION 37 AUTOMATION CELL - update and rank the classification leaderboard
if "classification_table" in globals():
    existing_classification_table_s37 = classification_table.copy()
elif "classification_leaderboard" in globals():
    existing_classification_table_s37 = classification_leaderboard.copy()
elif "classification_leaderboard_s36" in globals():
    existing_classification_table_s37 = classification_leaderboard_s36.copy()
elif "boosting_results_s36" in globals():
    existing_classification_table_s37 = boosting_results_s36.copy()
else:
    existing_classification_table_s37 = pd.DataFrame()

existing_classification_table_s37 = existing_classification_table_s37.rename(columns={
    "model": "Model", "Model Name": "Model", "accuracy": "Accuracy",
    "precision": "Precision", "recall": "Recall", "f1": "F1",
    "f1_score": "F1", "F1 Score": "F1", "roc_auc": "ROC AUC",
    "ROC_AUC": "ROC AUC", "AUC": "ROC AUC",
})
if "Rank" in existing_classification_table_s37.columns:
    existing_classification_table_s37 = existing_classification_table_s37.drop(columns="Rank")
if "Model" not in existing_classification_table_s37.columns:
    existing_classification_table_s37["Model"] = pd.Series(dtype=str)

existing_classification_table_s37 = existing_classification_table_s37[
    ~existing_classification_table_s37["Model"].astype(str).str.strip().str.lower().isin(
        ["mlp", "mlp classifier", "neural network", "neural-network classifier"]
    )
].copy()

all_columns_s37 = list(dict.fromkeys(
    list(existing_classification_table_s37.columns) + list(mlp_row_s37.columns)
))
for column_s37 in all_columns_s37:
    if column_s37 not in existing_classification_table_s37.columns:
        existing_classification_table_s37[column_s37] = np.nan
    if column_s37 not in mlp_row_s37.columns:
        mlp_row_s37[column_s37] = np.nan

classification_table_s37 = pd.concat(
    [existing_classification_table_s37[all_columns_s37], mlp_row_s37[all_columns_s37]],
    ignore_index=True,
)
sort_columns_s37 = [column for column in ["F1", "Recall", "ROC AUC", "Accuracy"]
                      if column in classification_table_s37.columns]
if sort_columns_s37:
    classification_table_s37 = classification_table_s37.sort_values(
        sort_columns_s37, ascending=False, na_position="last"
    ).reset_index(drop=True)
classification_table_s37.insert(0, "Rank", range(1, len(classification_table_s37) + 1))

classification_table = classification_table_s37.copy()
classification_leaderboard = classification_table_s37.copy()
classification_leaderboard_s37 = classification_table_s37.copy()
display(classification_table_s37.round(4))

In [ ]:
# SESSION 37 AUTOMATION CELL - compare MLP with boosting models
boosting_mask_s37 = classification_table_s37["Model"].astype(str).str.contains(
    "gradient boost|adaboost|ada boost", case=False, regex=True, na=False
)
mlp_mask_s37 = classification_table_s37["Model"].astype(str).str.strip().str.lower().eq(
    "mlp classifier"
)
mlp_vs_boosting_s37 = classification_table_s37[boosting_mask_s37 | mlp_mask_s37].copy()
display(mlp_vs_boosting_s37.round(4))

mlp_train_f1_s37 = f1_score(yctr_s37, mlp_train_predictions_s37, pos_label=0, zero_division=0)
mlp_test_f1_gap_s37 = mlp_train_f1_s37 - mlp_at_risk_f1_s37
print("MLP at-risk training F1:", round(mlp_train_f1_s37, 4))
print("MLP at-risk test F1:", round(mlp_at_risk_f1_s37, 4))
print("Training-minus-test at-risk F1 gap:", round(mlp_test_f1_gap_s37, 4))

In [ ]:
# SESSION 37 AUTOMATION CELL - random-seed stability analysis
stability_rows_s37 = []
for seed_s37 in [7, 21, 42, 99, 2026]:
    candidate_s37 = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=seed_s37),
    )
    with warnings.catch_warnings(record=True) as seed_warnings_s37:
        warnings.simplefilter("always", ConvergenceWarning)
        candidate_s37.fit(Xtr_f, yctr_s37)
    candidate_predictions_s37 = candidate_s37.predict(Xte_f)
    fitted_candidate_s37 = candidate_s37.named_steps["mlpclassifier"]
    stability_rows_s37.append({
        "Random Seed": seed_s37,
        "Accuracy": accuracy_score(ycte_s37, candidate_predictions_s37),
        "F1": f1_score(ycte_s37, candidate_predictions_s37, pos_label=1, zero_division=0),
        "At-Risk Recall": recall_score(ycte_s37, candidate_predictions_s37, pos_label=0, zero_division=0),
        "At-Risk F1": f1_score(ycte_s37, candidate_predictions_s37, pos_label=0, zero_division=0),
        "Iterations": fitted_candidate_s37.n_iter_,
        "Final Loss": float(fitted_candidate_s37.loss_),
        "Convergence Warnings": sum(
            issubclass(item.category, ConvergenceWarning) for item in seed_warnings_s37
        ),
    })

mlp_seed_stability_s37 = pd.DataFrame(stability_rows_s37)
display(mlp_seed_stability_s37.round(4))
print("Mean at-risk F1:", round(mlp_seed_stability_s37["At-Risk F1"].mean(), 4))
print("At-risk F1 standard deviation:", round(mlp_seed_stability_s37["At-Risk F1"].std(ddof=0), 4))
print("At-risk F1 range:", round(mlp_seed_stability_s37["At-Risk F1"].max() - mlp_seed_stability_s37["At-Risk F1"].min(), 4))

<!-- SESSION 37 AUTOMATION CELL -->
### Session 37 reflection

A neural network may be overfitting a small dataset when its training scores are substantially higher than its validation or test scores, training loss continues to fall while validation loss rises, or results change markedly across random seeds and cross-validation folds. Other warning signs include highly confident incorrect predictions, a large training-test F1 gap, convergence instability, and poorer unseen-data performance than simpler models such as logistic regression or boosting. These signs indicate that the network may be learning sample-specific noise rather than patterns that generalize to new students.

In [ ]:
# SESSION 37 AUTOMATION CELL - save and verify all output artifacts
results_dir_s37 = Path("results") / "session37"
results_dir_s37.mkdir(parents=True, exist_ok=True)

classification_table_path_s37 = results_dir_s37 / "session37_classification_table.csv"
mlp_row_path_s37 = results_dir_s37 / "session37_mlp_row.csv"
comparison_path_s37 = results_dir_s37 / "session37_mlp_vs_boosting.csv"
stability_path_s37 = results_dir_s37 / "session37_mlp_seed_stability.csv"

classification_table_s37.to_csv(classification_table_path_s37, index=False)
mlp_row_s37.to_csv(mlp_row_path_s37, index=False)
mlp_vs_boosting_s37.to_csv(comparison_path_s37, index=False)
mlp_seed_stability_s37.to_csv(stability_path_s37, index=False)

mlp_rows_s37 = classification_table_s37[
    classification_table_s37["Model"].astype(str).str.strip().str.lower().eq("mlp classifier")
]
if len(mlp_rows_s37) != 1:
    raise AssertionError(f"Expected exactly one MLP Classifier row; found {len(mlp_rows_s37)}")

required_metrics_s37 = [
    "Accuracy", "Precision", "Recall", "F1", "ROC AUC",
    "At-Risk Precision", "At-Risk Recall", "At-Risk F1",
]
values_s37 = mlp_rows_s37[required_metrics_s37].apply(pd.to_numeric, errors="coerce").to_numpy()
if not np.isfinite(values_s37).all() or not ((values_s37 >= 0) & (values_s37 <= 1)).all():
    raise AssertionError("The MLP row contains missing or invalid metrics.")

print("SESSION 37 OUTPUT ARTIFACT VERIFIED")
print("MLP at-risk F1:", round(mlp_at_risk_f1_s37, 4))
print("MLP ROC AUC:", round(mlp_roc_auc_s37, 4))
print("Convergence warnings:", mlp_convergence_warnings_s37)
print("Classification table path:", classification_table_path_s37)
print("MLP row path:", mlp_row_path_s37)
print("Boosting comparison path:", comparison_path_s37)
print("Seed stability path:", stability_path_s37)